In [1]:
import pandas as pd
from datetime import datetime

In [2]:
MESES = {
    "01": "enero", "02": "febrero", "03": "marzo", "04": "abril",
    "05": "mayo", "06": "junio", "07": "julio", "08": "agosto",
    "09": "septiembre", "10": "octubre", "11": "noviembre", "12": "diciembre"
}

In [5]:
df = pd.read_excel("columnaname.xlsx")  # Cambiar si el archivo tiene otro nombre real

# Inicializar listas para resultados
gacetas_array = []
origen_array = []
legislatura_array = []
fecha_iso_array = []
fecha_obj_array = []

for value in df['Nombre']:
    try:
        partes = value.split()
        fecha_str = partes[0]        # Ej: '19921215'
        legislatura = partes[1]      # Ej: 'I' o 'II'
        archivo = partes[-1]         # Ej: '0137_0008_S.pdf'
        tipo = archivo[-5] if '_' in archivo and archivo[-5] in ["C", "S", "B"] else ""

        # Partes de fecha
        anio, mes, dia = fecha_str[0:4], fecha_str[4:6], fecha_str[6:8]
        mes_texto = MESES.get(mes, "mes inválido")
        fecha_iso = f"{anio}{mes}{dia}"

        # Convertir a objeto datetime (para que Excel lo reconozca como fecha)
        fecha_dt = datetime.strptime(fecha_iso, "%Y%m%d").date() 

        # Descripción de gaceta
        numero_gaceta = archivo.split("_")[0]
        texto_gaceta = f"Gaceta Nro {numero_gaceta} del {dia} de {mes_texto} de {anio}"

        # Cámara o Senado
        origen = {"C": "Cámara", "S": "Senado"}.get(tipo, "")

        # Guardar
        gacetas_array.append(texto_gaceta)
        origen_array.append(origen)
        legislatura_array.append(legislatura)
        fecha_iso_array.append(fecha_iso)
        fecha_obj_array.append(fecha_dt)

    except Exception as e:
        print(f"Error procesando: {value} -> {e}")
        gacetas_array.append("")
        origen_array.append("")
        legislatura_array.append("")
        fecha_iso_array.append("")
        fecha_obj_array.append(None)

# Agregar columnas
df['GacetaDescripcion'] = gacetas_array
df['Origen'] = origen_array
df['Legislatura'] = legislatura_array
df['FechaFormatoISO'] = fecha_iso_array             # como string AAAAMMDD
df['FechaCompleta'] = fecha_obj_array               # como fecha real (datetime)

# Guardar archivo con fechas formateadas
df.to_excel("resultado_gacetas_v1.xlsx", index=False)


Error procesando: 19940541 III 0063_0016_S.pdf -> unconverted data remains: 1
Error procesando: 1999 1.pdf -> time data '1999' does not match format '%Y%m%d'
Error procesando: 20010832 X 0432_0016.pdf -> unconverted data remains: 2
Error procesando: a200_200_550_527_TD13_PL116_2005_S.pdf -> list index out of range
Error procesando: 20160732 XXV 549_32.pdf -> unconverted data remains: 2
Error procesando: 200_200_550_527_TD13_3_PL_142_1966_C.pdf -> list index out of range
